# This main.ipynb Notebook Highlights The Main Tests Performed

##  Overview

This notebook isolates the **most successful experiments** from `dev.ipynb` and presents them in a clean, structured format for evaluation and comparison.

##  What's Inside

### **Data Loading**
- Load pre-processed JSON files created during data exploration phase
- Files: `products_rag.json`, `relationships_rag.json`

### **4 Different RAG Models Tested**

#### **1. ModelRagDual** 🔵
- **Approach:** Cosine Similarity
- **Features:**
  - Dual embedding system (products + relationships)
  - Traditional semantic search using cosine similarity
  - No exclusion/filtering capabilities

#### **2. rag_dual_exclusion** 🟣
- **Approach:** Cosine Similarity + Intelligent Filtering
- **Features:**
  - Same dual embedding approach as ModelRagDual
  - **Automatic exclusion detection** (franchise filtering)
  - **Age-based filtering** (age-appropriate products)
  - LLM-powered query analysis

#### **3. ChromaDB (Traditional)** 🟢
- **Approach:** HNSW Algorithm (ChromaDB native)
- **Features:**
  - Leverages ChromaDB's built-in semantic search
  - Efficient HNSW (Hierarchical Navigable Small World) indexing
  - Fast approximate nearest neighbor search
  - No exclusion/filtering

#### **4. ChromaDB + Exclusions** 🟡
- **Approach:** HNSW + Intelligent Filtering
- **Features:**
  - ChromaDB's powerful semantic comparison
  - **Automatic exclusion detection**
  - **Age-based filtering**
  - Combines ChromaDB efficiency with smart filtering


# Load JSON models and create ChromeDB or Dataframe 

In [1]:
# Imports 
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from auxiliar_functions import create_chromadb_from_json, create_relationships_embeddings_dataframe, create_embeddings_dataframe
import json
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from typing import List, Dict

/Users/enlt-dlopes/Documents/Dev_Notebooks/Nadara/my_env_nadara/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load RAG - With or without Exclusion 
from rag_classes import DualEmbeddingNintendoRAG_exclusions, DualEmbeddingNintendoRAG, ChromaDBNintendoRAG, ChromaDBNintendoRAG_exclusions
# RAG ChromeDB without Exclusions
api_key = "REDACTED_SET_VIA_ENV_VAR" 

In [3]:
import json
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from typing import List, Dict

# Initialize SentenceTransformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Load products and relationships
with open('challenge/products_rag.json', 'r', encoding='utf-8') as f:
    products_rag = json.load(f)

with open('challenge/relationships_rag.json', 'r', encoding='utf-8') as f:
    relationships_rag = json.load(f)

print("✓ Loaded products:", len(products_rag['products']))
print("✓ Loaded relationships:", len(relationships_rag['relationships']))

✓ Loaded products: 16
✓ Loaded relationships: 238


In [4]:
# Head od Loade Products and Relationships
print("\nSample Products:")
for product in products_rag['products'][:2]:
    print(product)
print("\nSample Relationships:")
for relationship in relationships_rag['relationships'][:2]:
    print(relationship)


Sample Products:
{'id': 'prod_0001', 'text': 'Product: Nintendo Switch. Category: Console. Type: Unavailable. Franchise: Unavailable. Age rating: Unavailable. Release date: Unavailable. Total sales: 500,000. Available at: Store A, Store B, Store C.', 'metadata': {'name': 'Nintendo Switch', 'product_id': 1, 'category': 'Console', 'main_category': 'Console', 'type': None, 'franchise': None, 'min_age': None, 'release_date': None, 'times_sold': 500000, 'times_sold_Store_A': 200000, 'times_sold_Store_B': 150000, 'times_sold_Store_C': 150000, 'popularity_rank': 2, 'available_stores': ['Store A', 'Store B', 'Store C'], 'total_stock': 500000, 'is_available': True, 'store_strength_Store_A': 0.4, 'store_strength_Store_B': 0.3, 'store_strength_Store_C': 0.3}}
{'id': 'prod_0002', 'text': 'Product: Zelda: Breath of the Wild. Category: Games. Type: Adventure. Franchise: The Legend of Zelda. Age rating: 12. Released: 2017-03-03. Total sales: 250,000. Available at: Store A, Store B, Store C.', 'metad

# CHROM DB

In [5]:
# Chroma DB Approch to be tested
collection_chroma = create_chromadb_from_json(
    products_data=products_rag,
    collection_name="nintendo_products",
    persist_directory="./chroma_db"
)

print("\n✅ ChromaDB collection 'collection_chroma' is ready to use!")

✓ Deleted existing collection: nintendo_products
✓ Collection now contains 16 items
 CHROMADB COLLECTION CREATED SUCCESSFULLY
  • Collection name: nintendo_products
  • Storage location: ./chroma_db
  • Total products: 16
  • Embedding model: all-MiniLM-L6-v2
  • Embedding dimension: 384

✅ ChromaDB collection 'collection_chroma' is ready to use!


# DATAFRAME FOR COSINE SIM:

In [6]:
# Create the relationships embeddings DataFrame
model = SentenceTransformer('all-MiniLM-L6-v2')
df_embeddings = create_embeddings_dataframe(products_rag, model)

print(df_embeddings)
df_relationships_embeddings = create_relationships_embeddings_dataframe(
    relationships_rag, 
    model
)
print(df_relationships_embeddings)

Batches: 100%|██████████| 1/1 [00:00<00:00,  8.01it/s]


           id  product_id                              name  \
0   prod_0001           1                   Nintendo Switch   
1   prod_0002           2         Zelda: Breath of the Wild   
2   prod_0003           3       Zelda: Tears of the Kingdom   
3   prod_0004           4               Super Mario Odyssey   
4   prod_0005           5               Mario Kart 8 Deluxe   
5   prod_0006           6            Mario Party Superstars   
6   prod_0007           7                 Sonic Generations   
7   prod_0008           8                       Sonic Mania   
8   prod_0009           9     Animal Crossing: New Horizons   
9   prod_0010          10                        Splatoon 3   
10  prod_0011          11                          Pikmin 4   
11  prod_0012          12    Nintendo Switch Pro Controller   
12  prod_0013          13        Joy-Con Controllers (Pair)   
13  prod_0014          14          Nintendo Switch Dock Set   
14  prod_0015          15     Nintendo Switch Carrying 

Batches: 100%|██████████| 8/8 [00:00<00:00, 15.01it/s]

             id                                               text  \
0    rel_000001  Nintendo Switch is frequently bought together ...   
1    rel_000002  Nintendo Switch is frequently bought together ...   
2    rel_000003  Nintendo Switch is frequently bought together ...   
3    rel_000004  Nintendo Switch is frequently bought together ...   
4    rel_000005  Nintendo Switch is frequently bought together ...   
..          ...                                                ...   
233  rel_000234  Nintendo Switch Screen Protector is frequently...   
234  rel_000235  Nintendo Switch Screen Protector is frequently...   
235  rel_000236  Nintendo Switch Screen Protector is frequently...   
236  rel_000237  Nintendo Switch Screen Protector is frequently...   
237  rel_000238  Nintendo Switch Screen Protector is frequently...   

    product_a_id                    product_a_name product_a_category  \
0      prod_0001                   Nintendo Switch            Console   
1      prod_0

# COSINE SIM - RAGs MODEL

In [7]:
rag_dual = DualEmbeddingNintendoRAG(
    df_products=df_embeddings,
    df_relationships=df_relationships_embeddings,
    products_rag_path='challenge/products_rag.json',
    relationships_rag_path='challenge/relationships_rag.json',
    openai_api_key=api_key,
    model=model
)

rag_dual_exclusions = DualEmbeddingNintendoRAG_exclusions(
    df_products=df_embeddings,
    df_relationships=df_relationships_embeddings,
    products_rag_path='challenge/products_rag.json',
    relationships_rag_path='challenge/relationships_rag.json',
    openai_api_key=api_key,
    model=model,
    use_llm_for_exclusions=True
)

✅ Dual Embedding RAG System Initialized
✅ Using LLM-based exclusion detection
✅ Dual Embedding RAG System Initialized (with Exclusions)


# CHROME DB - RAGs MODEL

In [ ]:
# Initialize the ChromaDB RAG system
rag_chromadb = ChromaDBNintendoRAG(
    collection_chroma=collection_chroma,
    products_rag_path='challenge/products_rag.json',
    relationships_rag_path='challenge/relationships_rag.json',
    openai_api_key=api_key,
    model=model
)
# RAG CHROMA WITH EXCLSUIONS
rag_chromadb_exclusions = ChromaDBNintendoRAG_exclusions(
    collection_chroma=collection_chroma,
    products_rag_path='challenge/products_rag.json',
    relationships_rag_path='challenge/relationships_rag.json',
    openai_api_key=api_key,
    model=model,
    use_llm_for_exclusions=True
)

✅ ChromaDB RAG System Initialized
✅ Using LLM-based exclusion detection
✅ ChromaDB RAG System Initialized (with Exclusions)


# Exmplos Do email:
## -> Unrelated Queries: Example: "I want a pepperoni pizza with extra cheese please."
## -> Complex Queries: Example: "I want to buy a game for my nephew, at Store A, who is 5 years old. We loved Super Mario Odyssey, but I cannot buy a game from this family as he already has all Super Mario games."

In [9]:
exemplo_1 = "I want a pepperoni pizza with extra cheese please."
exemplo_2 = "I want to buy a game for my nephew, at Store A, who is 5 years old. We loved Super Mario Odyssey, but I cannot buy a game from this family as he already has all Super Mario games."

# Cosine App.

In [ ]:
print(" TESTING COSINE-SIM WITHOUT EXCLUSION HANDLING")
print(f"Query: {exemplo_1}")
results_dual = rag_dual.query(exemplo_1, top_k=5)

print("AGENT RESPONSE - WITHOUT EXCLUSIONS")
print(results_dual['response'])

print("\n" + "="*280)
print(f"Query 2: {exemplo_2}")
results_dual_2 = rag_dual.query(exemplo_2, top_k=5)

print("AGENT RESPONSE - WITHOUT EXCLUSIONS")
print(results_dual_2['response'])

 TESTING COSINE-SIM WITHOUT EXCLUSION HANDLING
Query: I want a pepperoni pizza with extra cheese please.

✅ Retrieved 5 products
✅ Retrieved 5 relationships

📝 AUGMENT: Building dual context...

Context Preview:
# User Query
User is asking: I want a pe

🤖 GENERATE: Calling GPT-4o-mini...
 Response generated (837 tokens)
AGENT RESPONSE - WITHOUT EXCLUSIONS
It seems like you’re looking for a pepperoni pizza with extra cheese, but I can only assist with Nintendo products. If you're interested in gaming, I recommend checking out the following popular Nintendo titles:

1. **Super Mario Odyssey** - A highly popular game suitable for ages 7 and up. It's a fantastic adventure that many players enjoy. Available at Store A, Store B, and Store C.

2. **Mario Kart 8 Deluxe** - Another great option for ages 6 and up, perfect for multiplayer fun. Available at Store A, Store B, and Store C.

3. **Mario Party Superstars** - Perfect for parties and family game nights, suitable for ages 6 and up. Availa

In [ ]:
print("\n" + "="*80)
print(" TESTING COSINE-SIM WITH EXCLUSION HANDLING")
print(f"Query: {exemplo_1}")
print("="*80)
results_dual_exclusions = rag_dual_exclusions.query(exemplo_1, top_k=5)

print("AGENT RESPONSE - WITH EXCLUSIONS")
print(results_dual_exclusions['response'])

print(f"Query 2: {exemplo_2}")
results_dual_exclusions_2 = rag_dual_exclusions.query(exemplo_2, top_k=5)

print("AGENT RESPONSE - WITH EXCLUSIONS")
print(results_dual_exclusions_2['response'])


 TESTING COSINE-SIM WITH EXCLUSION HANDLING
Query: I want a pepperoni pizza with extra cheese please.
   ✅ No exclusions detected

📝 AUGMENT: Building dual context with exclusions...
✅ Context built (1639 characters)

🤖 GENERATE: Calling GPT-4o-mini...
✅ Response generated (859 tokens)
AGENT RESPONSE - WITH EXCLUSIONS
I see you're interested in a pepperoni pizza with extra cheese, but I can't help with food orders. However, I can recommend some great Nintendo products for you!

Here are some suitable options based on your preferences:

1. **Super Mario Odyssey**
   - Age Rating: 7+
   - Available at: Store A, Store B, Store C

2. **Mario Party Superstars**
   - Age Rating: 6+
   - Available at: Store A, Store B, Store C

3. **Mario Kart 8 Deluxe**
   - Age Rating: 6+
   - Available at: Store A, Store B, Store C

For a fun game night, I recommend pairing **Mario Party Superstars** with **Mario Kart 8 Deluxe**. Both are family-friendly and perfect for multiplayer gaming!

If you also ne

In [18]:
#Only the responses
print(f"QUERY 1 -> {exemplo_1}")
print(f"COSINE W/OUT EXCLUSIONS:\n{results_dual_exclusions['response']}")
print("\n=================================\n")
print(f"COSINE W/ EXCLUSIONS:\n{results_dual['response']}")
print("\n\n\n")
print(f"Query2 -> {exemplo_2}")
print(f"COSINE W/OUT EXCLUSIONS:\n{results_dual_2['response']}")
print("\n=================================\n")
print(f"COSINE W/ EXCLUSIONS:\n{results_dual_exclusions_2['response']}")

QUERY 1 -> I want a pepperoni pizza with extra cheese please.
COSINE W/OUT EXCLUSIONS:
I see you're interested in a pepperoni pizza with extra cheese, but I can't help with food orders. However, I can recommend some great Nintendo products for you!

Here are some suitable options based on your preferences:

1. **Super Mario Odyssey**
   - Age Rating: 7+
   - Available at: Store A, Store B, Store C

2. **Mario Party Superstars**
   - Age Rating: 6+
   - Available at: Store A, Store B, Store C

3. **Mario Kart 8 Deluxe**
   - Age Rating: 6+
   - Available at: Store A, Store B, Store C

For a fun game night, I recommend pairing **Mario Party Superstars** with **Mario Kart 8 Deluxe**. Both are family-friendly and perfect for multiplayer gaming!

If you also need a way to carry your Nintendo Switch, consider the **Nintendo Switch Carrying Case**, available at Store A and Store C.

Let me know if you need more information or other recommendations!


COSINE W/ EXCLUSIONS:
It seems like you’re

# Chrome DB App.

In [23]:
print(" TESTING CHROME WITHOUT EXCLUSION HANDLING")
print(f"Query: {exemplo_1}")
results_chrome = rag_chromadb.query(exemplo_1, top_k=5)

print("AGENT RESPONSE - WITHOUT EXCLUSIONS")
print(results_dual['response'])

print("\n" + "="*280)
print(f"Query 2: {exemplo_2}")
results_chrome_2 = rag_chromadb.query(exemplo_2, top_k=5)

print("AGENT RESPONSE - WITHOUT EXCLUSIONS")
print(results_chrome_2['response'])

 TESTING CHROME WITHOUT EXCLUSION HANDLING
Query: I want a pepperoni pizza with extra cheese please.

📝 AUGMENT: Building context...
AGENT RESPONSE - WITHOUT EXCLUSIONS
It seems like you’re looking for a pepperoni pizza with extra cheese, but I can only assist with Nintendo products. If you're interested in gaming, I recommend checking out the following popular Nintendo titles:

1. **Super Mario Odyssey** - A highly popular game suitable for ages 7 and up. It's a fantastic adventure that many players enjoy. Available at Store A, Store B, and Store C.

2. **Mario Kart 8 Deluxe** - Another great option for ages 6 and up, perfect for multiplayer fun. Available at Store A, Store B, and Store C.

3. **Mario Party Superstars** - Perfect for parties and family game nights, suitable for ages 6 and up. Available at Store A, Store B, and Store C.

For a great gaming experience, consider pairing **Super Mario Odyssey** with **Mario Kart 8 Deluxe**. This combination is popular among players and ca

In [21]:
print(" TESTING CHROME WITHOUT EXCLUSION HANDLING")
print(f"Query: {exemplo_1}")
results_chrome_ex = rag_chromadb_exclusions.query(exemplo_1, top_k=5)

print("AGENT RESPONSE - WITHOUT EXCLUSIONS")
print(results_chrome_ex['response'])

print("\n" + "="*280)
print(f"Query 2: {exemplo_2}")
results_chrome_ex2 = rag_chromadb_exclusions.query(exemplo_2, top_k=5)

print("AGENT RESPONSE - WITHOUT EXCLUSIONS")
print(results_chrome_ex2['response'])

 TESTING CHROME WITHOUT EXCLUSION HANDLING
Query: I want a pepperoni pizza with extra cheese please.
   ✅ No exclusions detected

📝 AUGMENT: Building context with exclusions...

🤖 GENERATE: Calling GPT-4o-mini...
✅ Response generated (1009 tokens)
AGENT RESPONSE - WITHOUT EXCLUSIONS
It seems you're looking for a fun gaming experience! Here are some recommendations that are age-appropriate and popular:

1. **Super Mario Odyssey**
   - **Age Rating:** 7+
   - **Available at:** Store A, Store B, Store C
   - This game offers a vibrant adventure with Mario, perfect for a fun gaming session.

2. **Mario Party Superstars**
   - **Age Rating:** 6+
   - **Available at:** Store A, Store B, Store C
   - A fantastic party game that you can enjoy with friends and family.

3. **Mario Kart 8 Deluxe**
   - **Age Rating:** 6+
   - **Available at:** Store A, Store B, Store C
   - A thrilling racing game that’s great for competitive play.

To enhance your gaming setup, consider getting a **Nintendo Swit

In [24]:
#Only the responses
print(f"QUERY 1 -> {exemplo_1}")
print(f"Chrome W/OUT EXCLUSIONS:\n{results_chrome['response']}")
print("\n=================================\n")
print(f"Chrome W/ EXCLUSIONS:\n{results_chrome_ex['response']}")
print("\n\n\n")
print(f"Query2 -> {exemplo_2}")
print(f"Chrome W/OUT EXCLUSIONS:\n{results_chrome_2['response']}")
print("\n=================================\n")
print(f"Chrome W/ EXCLUSIONS:\n{results_chrome_ex2['response']}")

QUERY 1 -> I want a pepperoni pizza with extra cheese please.
Chrome W/OUT EXCLUSIONS:
It seems you’re looking for a pizza, which I can't help with. However, if you're interested in gaming, I have some great Nintendo product recommendations for you!

1. **Nintendo Switch** - A fantastic console with a wide variety of games. It's currently very popular and available at Store A, Store B, and Store C.

2. **Super Mario Odyssey** - An exciting game for the Nintendo Switch that's highly rated and sure to provide hours of fun. It's available at Store A, Store B, and Store C.

3. **Mario Party Superstars** - A fun party game perfect for playing with friends and family. This is also available at Store A, Store B, and Store C.

4. **Mario Kart 8 Deluxe** - Another crowd favorite that is great for both competitive and casual gaming. Available at Store A, Store B, and Store C.

**Complementary Product Combinations:**
- If you choose **Super Mario Odyssey**, consider pairing it with **Mario Kart 8

# Exemplos To Be Tested

In [19]:
# Examples
queries_10 = ["I would like to know the shipping status of my order.",
"What type of food is most suitable for my 5-year-old cat?",
"What can I buy to protect my son's console?",
"What game is most suitable for a child who likes animals?",
"I need a gift for a 15-year-old girl. I know she really likes store B.",
"A friend of mine loves films, but isn't really into fantasy. What game would you recommend?",
"My teenage son only likes the latest trends, what should I buy?", 
"I have a friend who is a collector. I'd like to give him an old game at a good price.", 
"My boyfriend and I like to play video games together. What would be a good gift?", 
"I wanted to give a complete video games kit to my 10-year-old son who loves sports." ]

In [32]:
# ============================================================================
# COMPREHENSIVE RAG TESTING LOOP
# ============================================================================

import pandas as pd
from datetime import datetime
import time

# Define all RAG models to test
rag_models = {
    # '🔵 Dual (Cosine)': rag_dual,
    '🟣 Dual + Exclusions': rag_dual_exclusions,
    # '🟢 ChromaDB (HNSW)': rag_chromadb,
    '🟡 ChromaDB + Exclusions': rag_chromadb_exclusions
}

# Test queries
queries_10 = [
    "I would like to know the shipping status of my order.",
    "I want a pepperoni pizza with extra cheese please.",
    "I want to buy a game for my nephew, at Store A, who is 5 years old. We loved Super Mario Odyssey, but I cannot buy a game from this family as he already has all Super Mario games.",
    "What game is most suitable for a child who likes animals?",
    "I need a gift for a 15-year-old girl. I know she really likes store B.",
    "A friend of mine loves films, but isn't really into fantasy. What game would you recommend?",
    "My boyfriend and I like to play video games together. What would be a good gift?", 
    "My teenage son only likes the latest trends, what should I buy?", 
    "I wanted to give a complete video games kit to my 10-year-old son who loves sports."
]

# Storage for results
all_results = []

# ============================================================================
# MAIN TESTING LOOP
# ============================================================================

print("\nSTARTING COMPREHENSIVE RAG EVALUATION")
print(f" Testing {len(rag_models)} RAG models across {len(queries_10)} queries")
print(f" Total tests: {len(rag_models) * len(queries_10)}")
print(f" Predicted number of tokens used...")
# 1k tokens per query approx.
print(f" {len(queries_10) * 1000} tokens")

start_time = time.time()


for query_idx, query in enumerate(queries_10, 1):
    print(f" QUERY {query_idx}/{len(queries_10)}")
    print(f" {query}")
    
    # Test each RAG model on this query
    for model_name, rag_model in rag_models.items():
        print(f" Testing: {model_name}")
        
        try:
            # Execute query
            query_start = time.time()
            result = rag_model.query(query, top_k=5)
            query_time = time.time() - query_start
            
            # Extract key metrics
            num_products = len(result.get('retrieved_products', []))
            num_relationships = len(result.get('retrieved_relationships', []) or result.get('relationships', []))
            response_length = len(result['response'])
            total_tokens = result['usage'].get('total_tokens', 0)
            
            # Check for exclusions (if available)
            exclusions_detected = result.get('auto_detected_exclusions', {})
            has_exclusions = any([
                exclusions_detected.get('franchises', []),
                exclusions_detected.get('products', []),
                exclusions_detected.get('categories', [])
            ])
            age_detected = exclusions_detected.get('age')
            
            # Print summary
            print(f"✅ Success!")
            print(f"   Query time: {query_time:.2f}s")
            print(f"   Products retrieved: {num_products}")
            print(f"   Relationships found: {num_relationships}")
            print(f"   Response length: {response_length} chars")
            print(f"   Tokens used: {total_tokens}")
            
            if has_exclusions or age_detected:
                print(f" Smart filtering active:")
                if exclusions_detected.get('franchises'):
                    print(f" Excluded franchises: {exclusions_detected['franchises']}")
                if age_detected:
                    print(f" Age filter: {age_detected} years")
            
            # Print first 200 chars of response
            print(f"\n   📄 Response preview:")
            print(f"   {result['response'][:200]}...")
            
            # Store result
            all_results.append({
                'query_number': query_idx,
                'query_text': query,
                'model_name': model_name,
                'success': True,
                'query_time': query_time,
                'num_products': num_products,
                'num_relationships': num_relationships,
                'response_length': response_length,
                'total_tokens': total_tokens,
                'excluded_franchises': len(exclusions_detected.get('franchises', [])),
                'age_detected': age_detected,
                'response': result['response'],
                'error': None
            })
            
        except Exception as e:
            print(f" Error: {str(e)}")
            all_results.append({
                'query_number': query_idx,
                'query_text': query,
                'model_name': model_name,
                'success': False,
                'query_time': None,
                'num_products': 0,
                'num_relationships': 0,
                'response_length': 0,
                'total_tokens': 0,
                'excluded_franchises': 0,
                'age_detected': None,
                'response': None,
                'error': str(e)
            })

total_time = time.time() - start_time

print(f" Total execution time: {total_time:.2f}s")
print(f" Total tests completed: {len(all_results)}")
print(f" Successful: {sum(1 for r in all_results if r['success'])}")
print(f" Failed: {sum(1 for r in all_results if not r['success'])}")

df_results = pd.DataFrame(all_results)



STARTING COMPREHENSIVE RAG EVALUATION
 Testing 2 RAG models across 9 queries
 Total tests: 18
 Predicted number of tokens used...
 9000 tokens
 QUERY 1/9
 I would like to know the shipping status of my order.
 Testing: 🟣 Dual + Exclusions
   ✅ No exclusions detected

📝 AUGMENT: Building dual context with exclusions...
✅ Context built (1630 characters)

🤖 GENERATE: Calling GPT-4o-mini...
✅ Response generated (934 tokens)
✅ Success!
   Query time: 7.38s
   Products retrieved: 5
   Relationships found: 5
   Response length: 1278 chars
   Tokens used: 934

   📄 Response preview:
   It seems you're inquiring about the shipping status of your order, which I cannot assist with directly. However, if you're interested in purchasing or learning about available Nintendo products, I can...
 Testing: 🟡 ChromaDB + Exclusions
   ✅ No exclusions detected

📝 AUGMENT: Building context with exclusions...

🤖 GENERATE: Calling GPT-4o-mini...
✅ Response generated (958 tokens)
✅ Success!
   Query time: 5.66

In [33]:
# ============================================================================
# PERFORMANCE COMPARISON
# ============================================================================

for model_name in rag_models.keys():
    model_results = df_results[df_results['model_name'] == model_name]
    
    if len(model_results) > 0:
        print(f"\n{model_name}")
        print(f"  Success rate: {model_results['success'].mean() * 100:.1f}%")
        print(f"  Avg query time: {model_results['query_time'].mean():.2f}s")
        print(f"  Avg products retrieved: {model_results['num_products'].mean():.1f}")
        print(f"  Avg relationships: {model_results['num_relationships'].mean():.1f}")
        print(f"  Avg tokens used: {model_results['total_tokens'].mean():.0f}")
        print(f"  Avg response length: {model_results['response_length'].mean():.0f} chars")

        # Exclusion stats (if applicable)
        if model_results['excluded_franchises'].sum() > 0:
            print(f"  Queries with exclusions detected: {(model_results['excluded_franchises'] > 0).sum()}")
        if model_results['age_detected'].notna().sum() > 0:
            print(f"  Queries with age detected: {model_results['age_detected'].notna().sum()}")

# ============================================================================
# SAVE RESULTS TO CSV
# ============================================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_filename = f"rag_evaluation_results_{timestamp}.xlsx"
df_results.to_excel(results_filename, index=False)

print(f"\n Results saved to: {results_filename}")

# ============================================================================
# QUERY-SPECIFIC ANALYSIS
# ============================================================================

print(" QUERY-SPECIFIC PERFORMANCE")

for query_idx in range(1, len(queries_10) + 1):
    query_results = df_results[df_results['query_number'] == query_idx]
    query_text = query_results.iloc[0]['query_text']

    # Best performing model (by token efficiency)
    successful_results = query_results[query_results['success']]
    if len(successful_results) > 0:
        best_model = successful_results.loc[successful_results['total_tokens'].idxmin()]
        print(f"  Most efficient: {best_model['model_name']} ({best_model['total_tokens']} tokens)")
        
        # Fastest model
        fastest_model = successful_results.loc[successful_results['query_time'].idxmin()]
        print(f"  Fastest: {fastest_model['model_name']} ({fastest_model['query_time']:.2f}s)")

print(" EVALUATION COMPLETE - Check df_results DataFrame for detailed data")


🟣 Dual + Exclusions
  Success rate: 100.0%
  Avg query time: 5.74s
  Avg products retrieved: 5.0
  Avg relationships: 5.0
  Avg tokens used: 900
  Avg response length: 908 chars
  Queries with exclusions detected: 1

🟡 ChromaDB + Exclusions
  Success rate: 100.0%
  Avg query time: 5.34s
  Avg products retrieved: 5.0
  Avg relationships: 15.0
  Avg tokens used: 993
  Avg response length: 913 chars
  Queries with exclusions detected: 1

 Results saved to: rag_evaluation_results_20251221_234522.xlsx
 QUERY-SPECIFIC PERFORMANCE
  Most efficient: 🟣 Dual + Exclusions (934 tokens)
  Fastest: 🟡 ChromaDB + Exclusions (5.66s)
  Most efficient: 🟣 Dual + Exclusions (912 tokens)
  Fastest: 🟡 ChromaDB + Exclusions (5.53s)
  Most efficient: 🟣 Dual + Exclusions (880 tokens)
  Fastest: 🟡 ChromaDB + Exclusions (4.30s)
  Most efficient: 🟣 Dual + Exclusions (836 tokens)
  Fastest: 🟡 ChromaDB + Exclusions (3.27s)
  Most efficient: 🟣 Dual + Exclusions (869 tokens)
  Fastest: 🟣 Dual + Exclusions (5.10s)
  M

# ALINHAMENTO FUTURO 

# Limitações e Estratégias Futuras

## Limitações Atuais

### Preocupações de Escalabilidade
Num cenário com milhares de produtos e centenas de relações (em vez de dezenas), a abordagem necessitaria de ajustes. Contudo, a lógica central de armazenamento de dados manteria-se similar:

**Estrutura de Dados**:
- Atribuir IDs únicos a cada produto com descrições e metadados armazenados numa base de dados
- Atribuir IDs únicos a cada relação entre 2 ou mais produtos com os respetivos metadados e descrições
- Indexar tabelas contendo embeddings e metadados para facilitar a pesquisa eficiente

**Fase de Augmentation**:
A etapa de seleção semântica torna-se crítica à escala. Os embeddings continuam eficazes para comparar relações, mas as estratégias de indexação devem ser otimizadas para desempenho - utilizando outro tipo de format de armazenamento não necessáriamente ChromDB.

### Tratamento de Exclusões
O sistema de deteção de exclusões não está totalmente resolvido. Casos adicionais podem surgir, exigindo:
- Etapa de pré-processamento para melhor discriminar e analisar queries
- Correspondência de padrões mais sofisticada para exclusões implícitas
- Análise LLM melhorada para cenários de restrição complexos

---

## Estratégias de Desenvolvimento Futuro

### 1. Sistema de Recomendação Sequencial
**Conceito**: Armazenar o histórico de interações do utilizador com o sistema RAG para fornecer recomendações contextuais em prompts futuros.

**Implementação**:
- Manter histórico de conversação por sessão de utilizador
- Construir janela de contexto a partir de queries e respostas anteriores
- Enriquecer o prompt com preferências históricas

### 2. Perfil de Utilizador e Rastreamento de Preferências
**Conceito**: Criar perfis de utilizador persistentes para personalizar as respostas do LLM.

**Benefícios**:
- Rastrear franchises, categorias e produtos preferidos
- Armazenar idade, restrições orçamentais e preferências de loja
- Fornecer recomendações mais direcionadas com base em interações passadas
- Todas estas informações cedidas ao LLM na fase de augmentation 

**Ponto de Integração**: Fase de Augmentation - onde decidimos que informação da nossa base de dados providenciar ao LLM.

### 3. Sistema de Feedback Loop
**Conceito**: Implementar mecanismos de feedback do utilizador para melhorar a qualidade das recomendações ao longo do tempo.

**Abordagens**:
- Feedback explícito (classificação de recomendações)
- Feedback implícito (taxas de clique, comportamento de compra)
- Refinamento contínuo do modelo com base na satisfação do utilizador

---

## Próximos Passos para Desenvolvimento

### Testes à Escala
**Objetivo**: Validar a abordagem com conjuntos de dados significativamente maiores para identificar gargalos de escalabilidade.

**Plano de Testes**:
- Aumentar o catálogo de produtos para mais de 1000 itens
- Expandir relações para mais de 500 conexões
- Monitorizar métricas de desempenho (tempo de query, uso de memória, consumo de tokens)
- Identificar onde os problemas de escalabilidade começam a surgir

### Otimização da Base de Dados
**Avaliação Atual**: O formato metadados + IDs + embeddings deve manter-se eficaz se bem implementado num sistema de base de dados.

**Incógnitas**:
- Degradação de desempenho com crescimento exponencial de dados
- Estratégias de indexação ótimas para pesquisa de embeddings em grande escala
- Trade-offs entre precisão e velocidade à escala
- Gestão de memória para matrizes de embedding grandes

### Considerações de Arquitetura
- Transição de DataFrames em memória para base de dados persistente (PostgreSQL com pgvector, Pinecone, Weaviate)
- Implementar camadas de cache para produtos frequentemente acedidos
- Considerar computação distribuída para geração de embeddings
- Explorar pesquisa híbrida combinando abordagens semânticas e baseadas em palavras-chave

---

## Resumo

A implementação RAG atual fornece uma base sólida, mas escalar para milhares de produtos requer:
- **Infraestrutura de dados melhorada** (base de dados adequada com capacidades de pesquisa vetorial)
- **Personalização aumentada** (perfis de utilizador, histórico de conversação)
- **Tratamento robusto de exclusões** (pré-processamento e análise LLM)

Todas as melhorias principais devem focar-se na **fase de Augmentation**, onde controlamos que contexto é fornecido ao LLM para geração. Também podemos experimentar com LLMs mais pesados já que esse passo é também fundamental 